In [3]:
import importlib

import ibis
import polars as pl
from ibis import deferred as _
from polars.testing import assert_frame_equal

import datasets
import datasets.fake
import pvm
import pvm.fields
import pvm.formulas
import pvm.pvm
from pvm.fields import Field, QuantityField, RateField
from pvm.pvm import PVM

importlib.reload(pvm.formulas)
importlib.reload(pvm.pvm)
importlib.reload(datasets)
importlib.reload(pvm)
importlib.reload(datasets.fake)
pass
pvm.__version__

'0.0.3'

In [4]:
datasets.sales.cost_effects_by_country_sku

country,sku,unit_cost_2020,cost_volume_2020,yield_rate_2020,raw_material_2020,cost_2020,unit_cost_2021,cost_volume_2021,yield_rate_2021,raw_material_2021,cost_2021,unit_cost__change__2020,cost_volume__change__2020,yield_rate__change__2020,raw_material__change__2020,cost__change__2020,unit_cost__effect__2020,cost_volume__effect__2020,yield_rate__effect__2020,raw_material__effect__2020
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Bouvet Island (Bouvetoya)""","""SKU0""",-13.451531,230.105074,0.854726,269.215071,-3095.265612,-7.144509,391.526218,0.864195,453.053255,-2797.262559,6.307022,161.421145,0.009469,183.838184,298.003053,2469.364638,-2171.361585,-57.705733,-2113.655852
"""Bouvet Island (Bouvetoya)""","""SKU1""",-64.884743,360.696298,0.861603,418.633801,-23403.686631,-62.604202,268.311532,0.832104,322.449494,-16797.429251,2.280542,-92.384766,-0.029499,-96.184307,6606.25738,611.895591,5994.361788,617.186379,5377.17541
"""Bouvet Island (Bouvetoya)""","""SKU2""",-64.251742,493.595726,0.859762,574.107309,-31714.385508,-61.009724,371.87468,0.831828,447.057095,-22687.971496,3.242019,-121.721047,-0.027934,-127.050214,9026.414012,1205.624664,7820.789349,802.3811,7018.408249
"""Bouvet Island (Bouvetoya)""","""SKU3""",-56.269671,387.563455,0.840516,461.101797,-21808.067958,-52.073185,241.714897,0.859624,281.186924,-12586.864583,4.196486,-145.848558,0.019107,-179.914873,9221.203375,1014.353075,8206.850299,-302.324465,8509.174764
"""Bouvet Island (Bouvetoya)""","""SKU4""",-12.905885,277.72621,0.864514,321.251196,-3584.302527,-11.938951,582.062675,0.849225,685.404768,-6949.217553,0.966934,304.336465,-0.015289,364.153572,-3364.915026,562.816388,-3927.731414,135.24666,-4062.978074
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Saint Vincent and the Grenadin…","""SKU0""",-9.663474,420.360844,0.839352,500.816029,-4062.146052,-10.584762,427.064947,0.851707,501.422667,-4520.380893,-0.921288,6.704103,0.012355,0.606637,-458.234841,-393.449915,-64.784926,-59.864458,-4.920468
"""Saint Vincent and the Grenadin…","""SKU1""",-64.987667,650.982093,0.861582,755.56587,-42305.807692,-66.075022,337.567476,0.840945,401.414325,-22304.778376,-1.087355,-313.414616,-0.020637,-354.151545,20001.029315,-367.055523,20368.084839,538.354893,19829.729945
"""Saint Vincent and the Grenadin…","""SKU2""",-60.930184,475.51546,0.859433,553.289595,-28973.244443,-62.330618,419.138726,0.859523,487.64097,-26125.1758,-1.400434,-56.376734,0.00009,-65.648625,2848.068643,-586.976132,3435.044775,-2.673328,3437.718102


In [43]:
price = RateField(
    "unit_price",
    definition=(_.volume * _.unit_price).sum() / _.volume.sum(),
    components=[
        QuantityField(
            "price_in_lc",
            definition=(_.volume * _.price_in_lc).sum() / _.volume.sum(),
        ),
        RateField(
            "fx_rate",
            definition=(_.volume * _.fx_rate * _.price_in_lc).sum() / (_.price_in_lc * _.volume).sum(),
        ),
    ],
)

revenue = Field(
    "revenue",
    definition=_.revenue.sum(),
    reconcile=False,
    components=[
        price,
        QuantityField(
            "volume",
            definition=_.volume.sum(),
        ),
        Field("flat_fee", definition=_.flat_fee.sum()),
    ],
)

In [44]:
con = ibis.polars.connect({"sales": datasets.sales.raw})
sales = con.table("sales")

obj = PVM().set_data(sales).set_periods(_.year, ["2020", "2021"]).set_hierarchy([_.customer, _.sku])

obj.set_graph(revenue)

In [ ]:
assert_frame_equal(
    obj.calculate_effects().to_polars().sort("customer", "sku"),
    datasets.sales.effects_by_customer_sku.sort("customer", "sku"),
    check_column_order=False,
    check_row_order=True,
    check_dtypes=False,
    atol=1e-2,
)

In [ ]:
unbound_table = ibis.table(sales.schema())
ibis.to_sql(obj.set_data(unbound_table).calculate_effects())

In [ ]:
obj.calculate_effects().to_polars().select(
    "sku",
    "country",
    "unit_price__effect__2020",
    "price_in_lc__effect__2020",
    "fx_rate__effect__2020",
).with_columns(
    (pl.col("unit_price__effect__2020") - pl.col("price_in_lc__effect__2020") - pl.col("fx_rate__effect__2020")).alias(
        "diff",
    ),
).sort(["country", "sku"])

sku,country,unit_price__effect__2020,price_in_lc__effect__2020,fx_rate__effect__2020,diff
str,str,f64,f64,f64,f64
"""SKU0""","""Bouvet Island (Bouvetoya)""",-245.058235,221.171414,-465.975594,-0.254054
"""SKU1""","""Bouvet Island (Bouvetoya)""",775.666603,3633.445595,-2856.300861,-1.478131
"""SKU2""","""Bouvet Island (Bouvetoya)""",1488.422966,4013.257933,-2523.560357,-1.27461
"""SKU3""","""Bouvet Island (Bouvetoya)""",-1632.4751,2424.875364,-4055.173563,-2.176901
"""SKU4""","""Bouvet Island (Bouvetoya)""",-1822.414223,-1480.970039,-341.218744,-0.225439
…,…,…,…,…,…
"""SKU0""","""Saint Vincent and the Grenadin…",-1.24942,509.947018,-511.49295,0.296511
"""SKU1""","""Saint Vincent and the Grenadin…",-1233.187334,2225.256196,-3460.371041,1.927511
"""SKU2""","""Saint Vincent and the Grenadin…",-244.626223,1678.173782,-1923.883343,1.083338
